# Stage 02 · Parallel Attestation Statistics

本 notebook 独立运行，只消费 stage 01 导出的 zip 包。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
from datetime import datetime

NOTEBOOK_NAME = "02_Parallel_Attestation_Statistics"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "main"
REPO_ROOT = Path("/content/ceg_wm_workspace")
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_Outputs_project_root"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
SCRIPT_PATH = REPO_ROOT / "scripts" / "02_Parallel_Attestation_Statistics.py"
ATTESTATION_ENV_PATH = DRIVE_PROJECT_ROOT / "secrets" / "attestation_env.json"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
INSPYRENET_WEIGHT_PATH = REPO_ROOT / "weights" / "InSPyReNet" / "InSPyReNet.pth"
INSPYRENET_WEIGHT_URL = "https://github.com/plemeri/InSPyReNet/releases/download/v1.0.0/InSPyReNet.pth"
MODEL_REPO_ID = "stabilityai/stable-diffusion-3.5-large"
SOURCE_PACKAGE_PATH = None

def run_checked(command, cwd=None, env=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def resolve_latest_stage_01_package():
    export_root = DRIVE_PROJECT_ROOT / "exports" / "01_Paper_Full_Cuda"
    candidates = sorted(export_root.glob("*.zip"), key=lambda item: item.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No stage 01 package found under {export_root}")
    return candidates[0]


In [ ]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
ensure_dir(DRIVE_PROJECT_ROOT)
ensure_dir(HF_HOME)
ensure_dir(INSPYRENET_WEIGHT_PATH.parent)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
    run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
    run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])


In [ ]:
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

run_checked(["python", "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

if not INSPYRENET_WEIGHT_PATH.exists():
    run_checked(["wget", "-O", str(INSPYRENET_WEIGHT_PATH), INSPYRENET_WEIGHT_URL])

from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL_REPO_ID, local_dir=str(REPO_ROOT / "models" / "sd3_5_large"), local_dir_use_symlinks=False)

if ATTESTATION_ENV_PATH.exists():
    env_payload = load_json(ATTESTATION_ENV_PATH)
    for key, value in env_payload.items():
        os.environ[str(key)] = str(value)

run_checked(["nvidia-smi"])


In [ ]:
stage_run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
source_package = Path(SOURCE_PACKAGE_PATH) if SOURCE_PACKAGE_PATH else resolve_latest_stage_01_package()
print(f"using source package: {source_package}")
command = [
    "python",
    str(SCRIPT_PATH),
    "--config",
    str(CONFIG_PATH),
    "--source-package",
    str(source_package),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--stage-run-id",
    stage_run_id,
]
run_checked(command, cwd=REPO_ROOT, env=os.environ.copy())

summary_path = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / stage_run_id / "stage_summary.json"
summary = load_json(summary_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(load_json(Path(summary["stage_manifest_path"])), ensure_ascii=False, indent=2))
print(json.dumps(load_json(Path(summary["package_manifest_path"])), ensure_ascii=False, indent=2))
